In [1]:
# get faq retriever
# create list queries
# get documents
# evaluation result (use precesion@k, recall@k, mrr, nDCG)

In [ ]:
import sys
sys.path.append("../../")

In [3]:
# get faq retriever
from pathlib import Path

from html import escape

import pandas as pd
from IPython.display import HTML, display

from evaluation.eval import evaluate_retrieval, load_tests, context_relevance
from evaluation.classes import ContextRelevanceWrapper
from evaluation.reporting import (
    build_eval_question_report,
    build_retrieval_doc_rows,
    display_eval_question_report,
    export_retrieval_report_to_markdown,
    get_node_text,
)
from retrieval.faq_retriever import search_faq_hybrid

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def retrieving(test, top_k):
    context = search_faq_hybrid(test.question, top_k=top_k)
    nodes = getattr(context, "nodes", context)
    return nodes

In [5]:
async def evaluate_all_retrieval(limit=None, top_k=5, markdown_path=None):
    """Render a readable retrieval report for the FAQ notebook."""

    tests = load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        nodes = retrieving(test, top_k)
        
        metrics = evaluate_retrieval(test, nodes, k=top_k)
        
        result = await context_relevance(ContextRelevanceWrapper(question=test.question, contexts=[get_node_text(item) for item in nodes]))
        metrics["context_relevance"] = result
        rows.append(metrics)
        filtered_metrics = {
            key: value
            for key, value in metrics.items()
            if key.startswith("precision") or key.startswith("recall") or key == "mrr" or key.startswith("ndcg") or key == "context_relevance"
        }
        doc_rows = build_retrieval_doc_rows(nodes, test.relevant_docs)
        display_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            metrics=filtered_metrics,
            text_blocks=[],
            doc_rows=doc_rows,
            index=index,
            total=len(selected_tests),
            metric_columns=4,
        )
        question_report = build_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            retrieved_docs=metrics["retrieved_docs"],
            doc_rows=doc_rows,
            extra_fields={"metrics": filtered_metrics},
            index=index,
            total=len(selected_tests),
        )
        question_reports.append(question_report)

    
    report_df = pd.DataFrame(rows)
    metric_columns = [column for column in report_df.columns if column.startswith(("precision", "recall", "ndcg")) or column in {"mrr", "context_relevance"}]
    summary_df = report_df[metric_columns].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_retrieval_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/08_faq_retrieve_evaluating.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df

In [6]:
await evaluate_all_retrieval(markdown_path="../reports/faq_retrieving_evaluation.md", top_k=10)

,rank,hit,section_id,title,score,preview
0,1,yes,83,How much refund you can get from the seller,0.9059,How much refund you can get from the seller Th...
1,2,no,77,I have been waiting a long time for my order d...,0.7621,I have been waiting a long time for my order d...
2,3,no,93,What to do if the seller does not issue a refund,0.7280,What to do if the seller does not issue a refu...
3,4,no,82,When the seller will issue a refund,0.6756,When the seller will issue a refund The seller...
4,5,no,77,I have been waiting a long time for my order d...,0.6368,are and how they work If you have not received...
5,6,no,70,Can I open a Discussion for a canceled order,0.6089,Can I open a Discussion for a canceled order Y...
6,7,no,94,I have received a defective product. What shou...,0.6048,I have received a defective product. What shou...
7,8,no,40,I am buying many products — which delivery met...,0.5993,I am buying many products — which delivery met...
8,9,no,41,How can I track a parcel?,0.5821,How can I track a parcel? You can track your p...
9,10,no,86,A debit card,0.5739,A debit card You will receive a refund to your...


,rank,hit,section_id,title,score,preview
0,1,yes,25,One delivery address,0.9885,One delivery address Select one delivery addre...
1,2,no,7,Can I have more than one HopShop account,0.8350,"Can I have more than one HopShop account Yes, ..."
2,3,no,40,I am buying many products — which delivery met...,0.7397,I am buying many products — which delivery met...
3,4,no,24,How to pay for orders made from multiple sellers,0.6821,How to pay for orders made from multiple selle...
4,5,no,26,How to choose a delivery option,0.6255,How to choose a delivery option If you buy at ...
5,6,no,47,The HopShop Delivery program — information for...,0.5514,The HopShop Delivery program — information for...
6,7,no,48,What you gain with HopShop Delivery,0.5248,What you gain with HopShop Delivery Convenient...
7,8,no,28,How to pay with one payment for orders from an...,0.5050,How to pay with one payment for orders from an...
8,9,no,77,I have been waiting a long time for my order d...,0.5000,I have been waiting a long time for my order d...
9,10,no,67,I want to change the delivery day. What should...,0.4924,I want to change the delivery day. What should...


,rank,hit,section_id,title,score,preview
0,1,yes,33,Why your payment is pending or canceled,1.0000,Why your payment is pending or canceled Your p...
1,2,no,34,Other reasons why your payment may be pending,0.7570,Other reasons why your payment may be pending ...
2,3,no,35,My payment has not gone through. How to retry ...,0.5757,My payment has not gone through. How to retry ...
3,4,no,29,Payment methods,0.4581,Payment methods You cannot link payment with i...
4,5,no,71,How to open a Discussion for a canceled order,0.4493,How to open a Discussion for a canceled order ...
5,6,no,42,Where to check the parcel tracking number and ...,0.4445,Where to check the parcel tracking number and ...
6,7,no,51,How to check the status of an HopShop Delivery...,0.4258,How to check the status of an HopShop Delivery...
7,8,no,32,HopShop gift cards,0.3963,HopShop gift cards If you are not sure what to...
8,9,no,30,How to split the payment,0.3587,How to split the payment In the Purchase Histo...
9,10,no,77,I have been waiting a long time for my order d...,0.3552,I have been waiting a long time for my order d...


,rank,hit,section_id,title,score,preview
0,1,yes,21,The seller's contact information,1.0000,The seller's contact information After complet...
1,2,no,41,How can I track a parcel?,0.7609,How can I track a parcel? You can track your p...
2,3,no,67,I want to change the delivery day. What should...,0.7249,I want to change the delivery day. What should...
3,4,no,66,I want to change the pick-up point or parcel l...,0.6751,I want to change the pick-up point or parcel l...
4,5,no,77,I have been waiting a long time for my order d...,0.6483,I have been waiting a long time for my order d...
5,6,no,77,I have been waiting a long time for my order d...,0.5740,are and how they work If you have not received...
6,7,no,20,Ask a question the seller,0.5620,Ask a question the seller If you want to ask t...
7,8,no,42,Where to check the parcel tracking number and ...,0.5590,Where to check the parcel tracking number and ...
8,9,no,68,The pick-up code does not work. What should I do?,0.5538,The pick-up code does not work. What should I ...
9,10,no,84,How you will learn about the refund,0.5067,How you will learn about the refund We will le...


,rank,hit,section_id,title,score,preview
0,1,yes,69,The parcel is damaged. What should I do?,1.0000,The parcel is damaged. What should I do? If yo...
1,2,no,66,I want to change the pick-up point or parcel l...,0.7360,I want to change the pick-up point or parcel l...
2,3,no,68,The pick-up code does not work. What should I do?,0.7031,The pick-up code does not work. What should I ...
3,4,no,94,I have received a defective product. What shou...,0.6785,I have received a defective product. What shou...
4,5,no,67,I want to change the delivery day. What should...,0.6213,I want to change the delivery day. What should...
5,6,no,77,I have been waiting a long time for my order d...,0.5066,I have been waiting a long time for my order d...
6,7,no,60,For what reasons you can file a complaint abou...,0.4329,For what reasons you can file a complaint abou...
7,8,no,63,Why can I not find a pick-up point or a parcel...,0.4279,Why can I not find a pick-up point or a parcel...
8,9,no,41,How can I track a parcel?,0.4221,How can I track a parcel? You can track your p...
9,10,no,74,What you should do if you do not receive your ...,0.4188,What you should do if you do not receive your ...


...

,rank,hit,section_id,title,score,preview
0,1,yes,96,Sending a product back to the seller,1.0000,Sending a product back to the seller If you se...
1,2,no,94,I have received a defective product. What shou...,0.4946,I have received a defective product. What shou...
2,3,no,60,For what reasons you can file a complaint abou...,0.4019,For what reasons you can file a complaint abou...
3,4,no,69,The parcel is damaged. What should I do?,0.3824,The parcel is damaged. What should I do? If yo...
4,5,no,59,How to file a complaint about an HopShop Deliv...,0.3751,How to file a complaint about an HopShop Deliv...
5,6,no,95,Deadline for processing a complaint,0.3733,Deadline for processing a complaint When the s...
6,7,no,97,Consumer sales (when a consumer buys from an e...,0.3601,Consumer sales (when a consumer buys from an e...
7,8,no,77,I have been waiting a long time for my order d...,0.2660,I have been waiting a long time for my order d...
8,9,no,82,When the seller will issue a refund,0.2636,When the seller will issue a refund The seller...
9,10,no,73,Purchases from private individuals or transact...,0.2488,Purchases from private individuals or transact...


,rank,hit,section_id,title,score,preview
0,1,yes,67,I want to change the delivery day. What should...,0.9899,I want to change the delivery day. What should...
1,2,no,77,I have been waiting a long time for my order d...,0.7970,I have been waiting a long time for my order d...
2,3,no,66,I want to change the pick-up point or parcel l...,0.7309,I want to change the pick-up point or parcel l...
3,4,no,78,Cancelling an order,0.6018,Cancelling an order You can cancel an order in...
4,5,no,77,I have been waiting a long time for my order d...,0.5755,are and how they work If you have not received...
5,6,no,41,How can I track a parcel?,0.5567,How can I track a parcel? You can track your p...
6,7,no,40,I am buying many products — which delivery met...,0.5507,I am buying many products — which delivery met...
7,8,no,70,Can I open a Discussion for a canceled order,0.5157,Can I open a Discussion for a canceled order Y...
8,9,no,46,Where you can see the estimated delivery time,0.4783,Where you can see the estimated delivery time ...
9,10,no,50,How to order a parcel with HopShop Delivery,0.4595,How to order a parcel with HopShop Delivery Wh...


,rank,hit,section_id,title,score,preview
0,1,no,67,I want to change the delivery day. What should...,0.9392,I want to change the delivery day. What should...
1,2,yes,77,I have been waiting a long time for my order d...,0.9175,I have been waiting a long time for my order d...
2,3,no,74,What you should do if you do not receive your ...,0.8346,What you should do if you do not receive your ...
3,4,yes,77,I have been waiting a long time for my order d...,0.7475,are and how they work If you have not received...
4,5,no,66,I want to change the pick-up point or parcel l...,0.7433,I want to change the pick-up point or parcel l...
5,6,no,68,The pick-up code does not work. What should I do?,0.6664,The pick-up code does not work. What should I ...
6,7,no,94,I have received a defective product. What shou...,0.6544,I have received a defective product. What shou...
7,8,no,40,I am buying many products — which delivery met...,0.6230,I am buying many products — which delivery met...
8,9,no,69,The parcel is damaged. What should I do?,0.6217,The parcel is damaged. What should I do? If yo...
9,10,no,70,Can I open a Discussion for a canceled order,0.5797,Can I open a Discussion for a canceled order Y...


,precision,recall,mrr,ndcg,context_relevance
mean,0.105,1.0,0.891,0.919,0.96


,question,relevant_docs,retrieved_docs,precision,recall,mrr,ndcg,context_relevance
0,Will I get a full refund including shipping if...,[83],"[83, 77, 93, 82, 70, 94, 40, 41, 86]",0.111111,1.0,1.000000,1.000000,0.50
1,Can I use multiple delivery addresses for one ...,[25],"[25, 7, 40, 24, 26, 47, 48, 28, 77, 67]",0.100000,1.0,1.000000,1.000000,0.50
2,Why does my payment status show as canceled or...,[33],"[33, 34, 35, 29, 71, 42, 51, 32, 30, 77]",0.100000,1.0,1.000000,1.000000,1.00
3,How can I find the seller's email and phone nu...,[21],"[21, 41, 67, 66, 77, 20, 42, 68, 84]",0.111111,1.0,1.000000,1.000000,1.00
4,What should I do if my parcel was damaged when...,[69],"[69, 66, 68, 94, 67, 77, 60, 63, 41, 74]",0.100000,1.0,1.000000,1.000000,1.00
5,How do I sign up using my Google or Facebook a...,[3],"[3, 1, 7, 2, 4, 68, 5, 54, 10, 64]",0.100000,1.0,1.000000,1.000000,1.00
6,How can I find my parcel's tracking number and...,[42],"[42, 51, 41, 77, 50, 59, 58, 64, 47, 49]",0.100000,1.0,1.000000,1.000000,1.00
7,How will I receive my refund from the seller?,[85],"[83, 85, 82, 84, 93, 77, 81, 74, 86]",0.111111,1.0,0.500000,0.630930,1.00
8,How long does it take to get a refund for a wi...,[91],"[91, 89, 93, 77, 34, 83, 86, 87, 90]",0.111111,1.0,1.000000,1.000000,1.00
9,How is the estimated delivery time calculated ...,[43],"[43, 44, 45, 46, 50, 37, 49, 47, 36, 48]",0.100000,1.0,1.000000,1.000000,1.00
